In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv('Datasets/clean_final_data.csv')
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 49222 entries, 0 to 49221
Data columns (total 17 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   OrderID          49222 non-null  int64  
 1   CustomerID       49222 non-null  int64  
 2   OrderDate        49222 non-null  str    
 3   ProductID        49222 non-null  int64  
 4   Quantity         49222 non-null  float64
 5   Discount         49222 non-null  float64
 6   PaymentMethod    49222 non-null  str    
 7   Status           49222 non-null  str    
 8   Age              49222 non-null  float64
 9   City             49222 non-null  str    
 10  SignupDate       49222 non-null  str    
 11  CustomerSegment  49222 non-null  str    
 12  ProductName      49222 non-null  str    
 13  Category         49222 non-null  str    
 14  UnitPrice        49222 non-null  float64
 15  Sales            49222 non-null  float64
 16  OrderValue       49222 non-null  float64
dtypes: float64(6), int64(3)

In [4]:
df

,OrderID,CustomerID,OrderDate,ProductID,Quantity,Discount,PaymentMethod,Status,Age,City,SignupDate,CustomerSegment,ProductName,Category,UnitPrice,Sales,OrderValue
0,500001,103695,2025-08-28,2003,4.0,10.0,Gateway,Completed,36.0,Qom,2024-03-05,Regular,USB-C Cable,Accessories,9.0,32.0,32.0
1,500002,107935,2024-05-31,2014,1.0,10.0,Wallet,Completed,49.0,Kerman,2025-06-04,Regular,Backpack,Accessories,42.0,38.0,38.0
2,500003,105141,2025-08-10,2002,1.0,20.0,CardToCard,Completed,36.0,Tehran,2025-10-19,Regular,Mechanical Keyboard,Electronics,62.0,50.0,50.0
3,500004,108780,2024-10-11,2012,1.0,10.0,Gateway,Completed,45.0,Shiraz,2023-12-07,Regular,Office Chair,Home Office,180.0,162.0,162.0
4,500005,101187,2024-02-19,2008,2.0,0.0,Gateway,Completed,24.0,Karaj,2023-12-02,Regular,Phone Case,Accessories,14.0,28.0,28.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
49217,549996,104649,2025-04-30,2013,2.0,5.0,CardToCard,Completed,32.0,Tehran,2025-03-24,Regular,Notebook,Stationery,7.0,13.0,13.0
49218,549997,108686,2025-01-26,2004,1.0,10.0,Gateway,Completed,45.0,Tehran,2024-03-08,New,Power Bank,Electronics,35.0,32.0,32.0
49219,549998,107639,2026-04-13,2010,1.0,0.0,Cash,Completed,56.0,Shiraz,2025-02-19,Regular,Fitness Band,Wearables,55.0,55.0,55.0
49220,549999,108215,2025-04-18,2005,1.0,0.0,CardToCard,Completed,20.0,Qom,2025-08-11,Regular,Laptop Stand,Accessories,28.0,28.0,28.0


In [5]:
# Date Engineering
def date_engineering(df):

    # convert date to time
    df['OrderDate'] = pd.to_datetime(df['OrderDate'])
    df['SignupDate'] = pd.to_datetime(df['SignupDate'])

    # Time features
    df['OrderMonth'] = df['OrdeDate'].dt.month #1-12
    df['OrderYear']= df['OrderDate'].dt.year   #1-31
    df['OrderDayOfWeek'] = df['OrderDate'].dt.dayofweek # 0 =Monday, 6 = Sunday
    df['OrderYear'] = df['OrderDate'].dt.year
    df['OrderQuarter'] = df['OrderDate'].dt.quarter # 1-4
    df['OrderWeekOfYear'] = df['OrderDate'].dt.isocalendar().week 

    # Boolean/Flags

    df['IsWeekend'] = df['OrderDayOfWeek'].isin([5,6]).astype(int)
    df['IsHolidaySeason'] = df['OrderMonth'].isin([11,12]).astype(int)
    df['IsSummer'] = df['OrderMonth'].isin([6,7,8]).astype(int)

    # Time since last order (if multiple orders per customer)
    # df['DaysSinceLastOrder'] = ...(advanced)

    # Day of year (for seasonal patterns)
    df['DayOfYear'] = df['OrderDate'].dt.dayofyear

    return df



In [6]:
df.columns

Index(['OrderID', 'CustomerID', 'OrderDate', 'ProductID', 'Quantity',
       'Discount', 'PaymentMethod', 'Status', 'Age', 'City', 'SignupDate',
       'CustomerSegment', 'ProductName', 'Category', 'UnitPrice', 'Sales',
       'OrderValue'],
      dtype='str')

In [7]:
drop_cols = ['OrderID', 'CustomerID', 'ProductID', 'OrderDate', 'SignupDate','ProductName','Sales','OrderValue','OrderDate','SignupDate']
target = 'CustomerSegment'

X= df.drop(columns=drop_cols + [target])
y= df[target]

from sklearn.preprocessing import LabelEncoder 
from  sklearn.model_selection import train_test_split

for col in X.select_dtypes(include=['object']).columns:
    X[col] = LabelEncoder().fit_transform(X[col])

y_enc = LabelEncoder().fit_transform(y)

X_train,X_test,y_train,y_test = train_test_split(X,y_enc,test_size = 0.2,random_state = 42)


C:\Users\HP\AppData\Local\Temp\ipykernel_11940\1069928471.py:10: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in X.select_dtypes(include=['object']).columns:


In [8]:

from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier(n_estimators =100,random_state = 42)
rf.fit(X_train,y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric

In [9]:
from sklearn.metrics import accuracy_score
print(accuracy_score(y_test,rf.predict(X_test)))

0.5040121889283901


In [10]:
print(f"Number of features: {X.shape[1]}")
print(f"Features: {X.columns.tolist()}")

Number of features: 8
Features: ['Quantity', 'Discount', 'PaymentMethod', 'Status', 'Age', 'City', 'Category', 'UnitPrice']


In [11]:
print(y.value_counts())
print(y.value_counts(normalize =True))

CustomerSegment
Regular    27296
New        17056
VIP         4870
Name: count, dtype: int64
CustomerSegment
Regular    0.554549
New        0.346512
VIP        0.098939
Name: proportion, dtype: float64


In [12]:
print(f"Total samples: {len(df)}")
print(f"features: {X.shape[1]}")
print(f"Target classes: {len(y.unique())}")


Total samples: 49222
features: 8
Target classes: 3


In [13]:
df.columns

Index(['OrderID', 'CustomerID', 'OrderDate', 'ProductID', 'Quantity',
       'Discount', 'PaymentMethod', 'Status', 'Age', 'City', 'SignupDate',
       'CustomerSegment', 'ProductName', 'Category', 'UnitPrice', 'Sales',
       'OrderValue'],
      dtype='str')

In [14]:
df['OrderDate'] = pd.to_datetime(df['OrderDate'])
df['SignupDate'] = pd.to_datetime(df['SignupDate'])
df['CustomerTenure'] = (df['OrderDate'] - df['SignupDate']).dt.days

customer_Stats = df.groupby('CustomerID').agg({
    'OrderID' :'count',
    'Sales':'sum',
}).reset_index()
customer_Stats.columns = ['CustomerID','OrderCount','TotalSpend']

# Merge back to add new columns

df = df.merge(customer_Stats, on = 'CustomerID',how = 'left')

# Create interaction columns

df['RevenuePerItem'] = df['Sales']/ df['Quantity']
df['IsDiscounted'] = (df['Discount'] > 0).astype(int)

# Now check your columns

print('\nYour New Columns')
print(df.columns.tolist())





Your New Columns
['OrderID', 'CustomerID', 'OrderDate', 'ProductID', 'Quantity', 'Discount', 'PaymentMethod', 'Status', 'Age', 'City', 'SignupDate', 'CustomerSegment', 'ProductName', 'Category', 'UnitPrice', 'Sales', 'OrderValue', 'CustomerTenure', 'OrderCount', 'TotalSpend', 'RevenuePerItem', 'IsDiscounted']


In [15]:
target  = 'CustomerSegment'

drop_cols = ['OrderID', 'CustomerID', 'ProductID',  
             'Sales', 'OrderValue', 'OrderDate', 'SignupDate']

drop_cols = [col for col in drop_cols + [target] if col in df.columns ]



X = df.drop(columns=drop_cols)
X = X.fillna(X.median(numeric_only=True))

y= df[target]

print('='*50)
print('FEATURES YOU ARE USING NOW:')
print('='*50)
#
print(f"Total features: {len(X.columns)}")
print(f"features : {X.columns.tolist()}")
print('\n' + '='*50)


FEATURES YOU ARE USING NOW:
Total features: 14
features : ['Quantity', 'Discount', 'PaymentMethod', 'Status', 'Age', 'City', 'ProductName', 'Category', 'UnitPrice', 'CustomerTenure', 'OrderCount', 'TotalSpend', 'RevenuePerItem', 'IsDiscounted']



In [16]:
df.columns

Index(['OrderID', 'CustomerID', 'OrderDate', 'ProductID', 'Quantity',
       'Discount', 'PaymentMethod', 'Status', 'Age', 'City', 'SignupDate',
       'CustomerSegment', 'ProductName', 'Category', 'UnitPrice', 'Sales',
       'OrderValue', 'CustomerTenure', 'OrderCount', 'TotalSpend',
       'RevenuePerItem', 'IsDiscounted'],
      dtype='str')

In [21]:
# # Encoding Techniques
categorical_cols = X.select_dtypes(include=['object']).columns
print(f"Categorical columns : {categorical_cols.tolist()}" )

from sklearn.preprocessing import LabelEncoder
for col in categorical_cols:
    X[col] = LabelEncoder().fit_transform(X[col].astype(str))
    print(f" Encoded {col}")

target_encoder = LabelEncoder()
y_encoded = target_encoder.fit_transform(y)
# y_encoded = target_encoder

print(f"Target Classes : {y_encoded}")

Categorical columns : ['PaymentMethod', 'Status', 'City', 'ProductName', 'Category']
 Encoded PaymentMethod
 Encoded Status
 Encoded City
 Encoded ProductName
 Encoded Category
Target Classes : [1 1 1 ... 1 1 0]


C:\Users\HP\AppData\Local\Temp\ipykernel_11940\3420417828.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X.select_dtypes(include=['object']).columns


In [29]:

from sklearn.model_selection import train_test_split

X_train,X_test,y_train,y_test = train_test_split(X,y_encoded,test_size=0.2,random_state=42,stratify=y_encoded)
print(f"\nTraining: {X_train.shape[0]} samples")
print(f"Test: {X_test.shape[0]} samples")
# #
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators =150,max_depth=150,min_samples_split=5,min_samples_leaf=2,random_state=42,n_jobs= -1)

rf.fit(X_train,y_train)




print(list(rf.feature_names_in_))
print(list(X_test.columns))


Training: 39377 samples
Test: 9845 samples
['Quantity', 'Discount', 'PaymentMethod', 'Status', 'Age', 'City', 'ProductName', 'Category', 'UnitPrice', 'CustomerTenure', 'OrderCount', 'TotalSpend', 'RevenuePerItem', 'IsDiscounted']
['Quantity', 'Discount', 'PaymentMethod', 'Status', 'Age', 'City', 'ProductName', 'Category', 'UnitPrice', 'CustomerTenure', 'OrderCount', 'TotalSpend', 'RevenuePerItem', 'IsDiscounted']


In [30]:
y_ored = rf.predict(X_test)

from sklearn.metrics import accuracy_score,classification_report,confusion_matrix

print(accuracy_score(y_test,y_ored))

0.5946165566277298


In [31]:
print("Classification Report")
print(classification_report(y_test,y_ored,target_names=target_encoder.classes_))

Classification Report
              precision    recall  f1-score   support

         New       0.66      0.18      0.28      3411
     Regular       0.59      0.96      0.73      5460
         VIP       1.00      0.01      0.02       974

    accuracy                           0.59      9845
   macro avg       0.75      0.38      0.34      9845
weighted avg       0.65      0.59      0.50      9845



In [ ]:
print("\nConfusion Matrix")
print(confusion_matrix(y_test,y_ored))




Confusion Matrix
[[ 609 2802    0]
 [ 224 5236    0]
 [  87  878    9]]
